In [1]:
import pandas as pd
import tsam
import plotly.express as px

# General pipeline
8760 hourly rows => 365 daily profiles => cluster similar days => representative days + weights + assignments

# Data Loading

## Separator:
Depending on the separator of the data in the CSV files, you might want to use 

In [15]:
demand_UA = pd.read_csv("data/Demand_2025_UA_90TWh.csv", sep=";")
demand_UA

,snapshot,UA
0,01.01.2025 00:00,9933.090245
1,01.01.2025 01:00,9888.972586
2,01.01.2025 02:00,10042.281440
3,01.01.2025 03:00,10271.141780
4,01.01.2025 04:00,10615.259500
...,...,...
8755,31.12.2025 19:00,12003.311250
8756,31.12.2025 20:00,11252.759630
8757,31.12.2025 21:00,10689.156570
8758,31.12.2025 22:00,10086.399100


In [3]:
demand_entso = pd.read_csv("data/Demand_ENTSO_E.csv", sep=";")
demand_entso

,snapshot,AL,AT,BA,BE,BG,CH,CZ,DE,DK,...,NL,NO,PL,PT,RO,RS,SE,SI,SK,UK
0,01.01.2025 00:00,459,3871,609,6691,2374,4603,4029,28833,2606,...,9796,11525,11852,3591,3645,2578,9701,728,1823,18285
1,01.01.2025 01:00,428,3717,571,6327,2292,4711,3989,27781,2511,...,9535,11586,11284,3480,3532,2474,9551,699,1780,18154
2,01.01.2025 02:00,396,3522,539,6128,2237,4861,3885,26917,2426,...,9266,11540,10983,3289,3467,2316,9377,666,1733,16545
3,01.01.2025 03:00,376,3488,523,5982,2206,4810,3855,27016,2400,...,9013,11517,10827,3126,3438,2175,9417,648,1729,14918
4,01.01.2025 04:00,367,3598,526,5967,2194,4725,3865,26870,2405,...,9040,11602,10754,3007,3428,2059,9560,664,1724,14200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,31.12.2025 19:00,623,4743,913,7678,2916,5301,4831,32565,2788,...,10804,11969,14245,4818,4350,3474,10638,932,2096,27044
8756,31.12.2025 20:00,589,4518,871,7462,2826,5092,4782,31431,2639,...,10185,11741,13609,4395,4166,3266,10310,860,2032,25212
8757,31.12.2025 21:00,558,4563,817,7659,2792,4918,4688,31197,2533,...,9827,11408,13228,4050,4041,3164,10173,813,2017,23374
8758,31.12.2025 22:00,539,4362,759,7697,2701,4948,4497,30146,2438,...,9638,11166,12693,3893,3924,3157,9817,773,2028,20966


In [4]:
hydro_inflow_scaled_2025 = pd.read_csv("data/hydro_inflow_scaled_2025.csv", sep=';')
hydro_inflow_scaled_2025

,snapshot,AT_hydro_2025,BA_hydro_2025,BE_hydro_2025,BG_hydro_2025,CH_hydro_2025,CZ_hydro_2025,DE_hydro_2025,ES_hydro_2025,FI_hydro_2025,...,RO_hydro_2025.1,RS_hydro_2025,RS_hydro_2025.1,RS_hydro_2025.2,SI_hydro_2025,UA_hydro_2025,UK_hydro_2025,XK_hydro_2025,SE_hydro_2025,AL_hydro_2025
0,01.01.2025 00:00,230,83,2,49,411,39,54,922,420,...,56,2,2,8,135,556,43,3,4570,155
1,01.01.2025 01:00,229,83,2,49,411,40,54,922,420,...,56,2,2,8,135,555,43,3,4569,152
2,01.01.2025 02:00,229,83,2,49,412,41,54,922,421,...,56,2,2,8,135,555,43,3,4562,151
3,01.01.2025 03:00,228,83,2,49,412,41,54,922,422,...,56,2,2,8,135,555,43,3,4555,151
4,01.01.2025 04:00,228,83,2,49,412,41,54,922,423,...,56,2,2,8,135,555,43,3,4545,150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,31.12.2025 19:00,247,339,10,38,726,41,139,4620,501,...,45,10,14,7,476,446,63,5,6914,810
8756,31.12.2025 20:00,247,339,10,38,726,41,139,4606,501,...,45,10,14,6,475,445,63,5,6914,809
8757,31.12.2025 21:00,247,338,10,38,726,41,139,4592,501,...,45,10,14,6,475,444,63,5,6913,808
8758,31.12.2025 22:00,247,338,10,38,725,41,138,4579,501,...,45,10,14,6,474,443,63,5,6913,807


In [6]:
ror_cf = pd.read_csv("data/ror_capacity_factors.csv", sep=',')
ror_cf

,snapshot,AL_ror_2025,AT_ror_2025,BA_ror_2025,BE_ror_2025,BG_ror_2025,CH_ror_2025,CZ_ror_2025,DE_ror_2025,EE_ror_2025,...,NO_ror_2025,PL_ror_2025,PT_ror_2025,RO_ror_2025,RS_ror_2025,SE_ror_2025,SI_ror_2025,SK_ror_2025,UK_ror_2025,XK_ror_2025
0,01.01.2025 00:00,0.06198,0.304260,0.113561,0.687213,0.132907,0.266242,0.128353,0.596077,0.000,...,0.592028,0.503166,0.140826,0.371941,0.428571,0.35421,0.090094,0.232892,0.19516,0.07694
1,01.01.2025 01:00,0.06074,0.300330,0.149166,0.687989,0.132323,0.264119,0.406737,0.591670,0.000,...,0.585834,0.503452,0.062204,0.369386,0.411529,0.35409,0.087915,0.231347,0.19512,0.07694
2,01.01.2025 02:00,0.06031,0.300783,0.155915,0.626787,0.129821,0.262051,0.430766,0.584802,0.000,...,0.578365,0.502625,0.059455,0.363477,0.338847,0.35360,0.087664,0.233002,0.19510,0.07682
3,01.01.2025 03:00,0.06016,0.306833,0.111967,0.609455,0.130377,0.260064,0.438508,0.577704,0.000,...,0.576665,0.502492,0.044515,0.359085,0.249123,0.35307,0.087212,0.230574,0.19506,0.07675
4,01.01.2025 04:00,0.05997,0.305347,0.111861,0.587503,0.135743,0.258158,0.438407,0.575100,0.000,...,0.580272,0.502750,0.096728,0.359085,0.246115,0.35229,0.087112,0.230022,0.19502,0.07684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,31.12.2025 19:00,0.32361,0.299895,0.715166,0.084536,0.405431,0.221143,0.384969,0.369924,0.000,...,0.598035,0.648884,0.428867,0.478301,0.574937,0.53590,0.534267,0.533775,0.28359,0.15014
8756,31.12.2025 20:00,0.32327,0.288157,0.732278,0.084367,0.288991,0.226015,0.400188,0.365093,0.000,...,0.594231,0.608019,0.846472,0.467601,0.358396,0.53586,0.350906,0.374724,0.28357,0.15002
8757,31.12.2025 21:00,0.32293,0.268775,0.733075,0.091988,0.238842,0.231023,0.423010,0.362756,0.000,...,0.590426,0.513844,0.749592,0.453068,0.213534,0.53581,0.244475,0.305960,0.28355,0.14995
8758,31.12.2025 22:00,0.32259,0.252255,0.719471,0.130058,0.197775,0.236112,0.422731,0.360581,0.625,...,0.583911,0.508509,0.356823,0.397413,0.192982,0.53577,0.109364,0.233002,0.28354,0.14994


In [8]:
solar_cf = pd.read_csv("data/solar_capacity_factors.csv", sep=';')
solar_cf

,snapshot,AL_solar_2025,AT_solar_2025,BA_solar_2025,BE_solar_2025,BG_solar_2025,CH_solar_2025,CZ_solar_2025,DE_solar_2025,DK_solar_2025,...,NO_solar_2025,PL_solar_2025,PT_solar_2025,RO_solar_2025,RS_solar_2025,SE_solar_2025,SI_solar_2025,SK_solar_2025,UA_solar_2025,XK_solar_2025
0,01.01.2025 00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,01.01.2025 01:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,01.01.2025 02:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,01.01.2025 03:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,01.01.2025 04:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,31.12.2025 19:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8756,31.12.2025 20:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8757,31.12.2025 21:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8758,31.12.2025 22:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Duplicates

In [12]:
data = [demand_UA, demand_entso, hydro_inflow_scaled_2025, ror_cf, solar_cf]

print(sum(df.duplicated().sum() for df in data))

0
